# 10_masking_redesign

원본 Colab 셀에서 분리. (`#@title [재설계] 조합 단위 마스킹 - 타겟 전부 살리고 비타겟만 가림`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [재설계] 조합 단위 마스킹 - 타겟 전부 살리고 비타겟만 가림
import os, pickle                                             # 경로 도구, 인덱스(pkl) 로드 도구
import pandas as pd                                            # CSV 로드 도구
import numpy as np                                              # 픽셀 배열·배경색 계산 도구
from PIL import Image                                            # 이미지 로드·저장 도구

TL_DIR = "/content/drive/MyDrive/1팀 공유 문서/김태윤"              # 작업 루트
COLLECT = f"{TL_DIR}/collected_56_18"                            # 기존 타겟별 복제 폴더(파일명 동일)
OUT = f"{TL_DIR}/masked_combo_56_18"                             # 새 결과 폴더: 조합당 이미지 1장
os.makedirs(OUT, exist_ok=True)

with open(f"{TL_DIR}/combo_boxes_index.pkl", "rb") as f:          # (조합,각도) → 그 이미지 전체 알약 박스
    combo_boxes = pickle.load(f)

master = pd.read_csv(f"{TL_DIR}/tl_search_results_56_18/master_matches.csv")  # 69개 타겟 매칭표
TARGET_DLS = set(master["tl_dl_idx"].unique().tolist())          # 69개 타겟의 TL 기준 dl_idx 집합

PAD = 8                                                          # 마스킹 시 박스 여유 픽셀

def mask_non_targets(img_arr, boxes, target_dls):                # 타겟 아닌 알약만 배경색으로 덮는 함수
    for dl, box in boxes:                                        # 이 이미지 속 알약 하나씩
        if dl in target_dls:                                     # 타겟이면 손대지 않음
            continue
        x, y, w, h = map(int, box)                                # 비타겟 박스 좌표
        x0, y0 = max(x-PAD, 0), max(y-PAD, 0)                     # 좌상단(패딩 적용)
        x1 = min(x+w+PAD, img_arr.shape[1])                       # 우측 경계
        y1 = min(y+h+PAD, img_arr.shape[0])                       # 하단 경계
        top = img_arr[max(y0-5,0):y0, x0:x1].reshape(-1,3)        # 박스 위쪽 배경 픽셀 샘플
        bot = img_arr[y1:min(y1+5,img_arr.shape[0]), x0:x1].reshape(-1,3)  # 아래쪽 배경 샘플
        ring = np.concatenate([top, bot]) if len(top)+len(bot) else np.empty((0,3))
        bg = np.median(ring, axis=0) if len(ring) else np.array([128,128,128])  # 주변색 중앙값
        img_arr[y0:y1, x0:x1] = bg.astype(np.uint8)                # 배경색으로 덮기
    return img_arr

# [1] 기존 타겟별 폴더에 중복 복사된 파일들을 파일명 기준으로 1개씩만 수집
seen = {}                                                          # {파일명: 원본경로}
for folder in os.listdir(COLLECT):
    fp = os.path.join(COLLECT, folder)
    if not os.path.isdir(fp): continue
    for fn in os.listdir(fp):
        if not fn.endswith((".png",".jpg")): continue
        seen.setdefault(fn, os.path.join(fp, fn))                  # 처음 발견한 경로만 사용(내용 동일)

print(f"고유 이미지 수: {len(seen)}")

# [2] 고유 이미지마다 비타겟만 마스킹 → 조합당 1장으로 저장
coverage = {}                                                      # {타겟dl: 등장 이미지 수} 참고용
for fn, src_path in seen.items():
    combo = fn.split("_")[0]                                       # 조합 prefix
    parts = fn.split("_")
    angle = parts[5] if len(parts) > 5 else ""                     # 각도
    boxes = combo_boxes.get((combo, angle), [])                    # 이 이미지 전체 박스
    if not boxes:
        continue

    img = np.array(Image.open(src_path).convert("RGB"))            # 이미지 로드
    img = mask_non_targets(img, boxes, TARGET_DLS)                 # 비타겟만 마스킹
    Image.fromarray(img).save(os.path.join(OUT, fn))               # 저장(중복 없이 1장)

    for dl, _ in boxes:                                             # 이 이미지의 타겟들 집계
        if dl in TARGET_DLS:
            coverage[dl] = coverage.get(dl, 0) + 1

print(f"저장 완료 → {OUT}")
cov_df = pd.DataFrame(sorted(coverage.items()), columns=["tl_dl_idx","image_count"])
cov_df = cov_df.merge(master[["tl_dl_idx","dl_name","src"]].drop_duplicates(), on="tl_dl_idx", how="left")
print(cov_df.to_string(index=False))                               # 타겟별 등장 장수 확인(42±허용)